# Lecture 6.3: Forking Sessions to Explore Different Approaches

In the previous lecture you saw that **resuming** a session extends one conversation. That is exactly what you want when you are continuing a task. It is the wrong tool when you want to ask "what if we did this differently?", because the detour becomes part of the history for good.

**Forking** solves that. A fork starts with a copy of the original conversation and then diverges. The original stays exactly as it was.

In this notebook you will:

1. Build a base conversation that settles a few facts for a seed swap day.
2. Fork it twice online, so two different follow-ups start from the same point.
3. Prove the branches are independent, and see that **files are not branched**.
4. Fork a session **offline** with the standalone `fork_session()` function, branching from an earlier point.

### Cell map

Every cell in this notebook is numbered. Follow the numbers as you work.

| Cell | What it does |
| :-- | :-- |
| 1 | Install the SDK |
| 2 | Set up the API key |
| 3 | Choose the model |
| 4 | Create the Seed Library files and record their hashes |
| 5 | Verify the files |
| 6 | Define the prompts |
| 7 | Base session, turn 1 |
| 8 | Base session, turn 2 |
| 9 | Fork A |
| 10 | Fork B |
| 11 | Ground truth: IDs, files, session count |
| 12 | Independence: what each branch remembers |
| 13 | Files are shared across branches |
| 14 | Offline fork with `fork_session()` |
| 15 | Resume the offline fork |
| 16 | Final verification |

Agent output varies between runs. That is normal for a real agent. In each cell, look for the behavior described, not for identical wording.

## Cell 1: Install the Claude Agent SDK

This cell installs the `claude-agent-sdk` Python package, the only package this notebook needs. The SDK bundles the Claude Code CLI, so there is nothing else to install.

| Part | Meaning |
| :-- | :-- |
| `==0.2.167` | Pins one exact SDK version so every example behaves as shown in the course. |
| `-q` | Quiet mode, which keeps the install output short. |

If the package is already present in your current Colab session, pip checks it and finishes quickly. If you see a message asking you to restart the runtime, do that and then continue from this cell.

In [ ]:
# Cell 1
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.167 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.2 MB/s eta 0:00:00


## Cell 2: Set up your API key

The agent needs an Anthropic API key. In Colab, the safe way to provide it is **Colab Secrets**, so the key never appears in your notebook.

**Create the key:** sign in to the Claude Console at https://platform.claude.com/ and open **API Keys** to create one.

**Add it to Colab Secrets:**

1. Click the **key icon** in the left sidebar.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value.
5. Switch **Notebook access** on.

The code reads the secret with `userdata.get("ANTHROPIC_API_KEY")` and writes it to the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

**Running locally instead of Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before starting Python.

In [ ]:
# Cell 2
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

Every agent call in this notebook uses the `MODEL_NAME` variable defined here. We use Claude Haiku 4.5, which is fast and inexpensive, and more than capable for these demos.

Because every `ClaudeAgentOptions` below passes `model=MODEL_NAME`, **changing this one variable changes the model for the entire notebook**. Model names change over time, so the comment in the cell links to the models page where you can find the latest ones.

In [ ]:
# Cell 3
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4: Create the Seed Library files

Our scenario is a community **Seed Library** preparing a seed swap day. This cell creates two small files under `/content/seedlibrary_demo/`:

| File | Contents |
| :-- | :-- |
| `catalog.txt` | Six seed varieties with packet counts. |
| `notes.txt` | Short swap-day logistics, including the time doors open. |

It then computes a **SHA-256 hash** of each file and stores them in `ORIGINAL_HASHES`. A hash is a fingerprint of a file's exact contents. If even one character changes, the hash changes. We will compare against these hashes at the end to prove that none of the forking in this notebook touched these two files.

Notice what is **not** in these files: any table number. That fact will exist only in a prompt. This matters later, because it lets us tell exactly where each session learned something.

The small `sha256_of` helper is reused in Cells 11 and 16.

In [ ]:
# Cell 4
import hashlib
import os

DEMO_DIR = "/content/seedlibrary_demo"
os.makedirs(DEMO_DIR, exist_ok=True)

catalog = (
    "marigold: 40 packets\n"
    "basil: 12 packets\n"
    "okra: 75 packets\n"
    "sunflower: 30 packets\n"
    "cilantro: 22 packets\n"
    "zinnia: 18 packets\n"
)

notes = (
    "Seed swap day notes\n"
    "Doors open at 10:00\n"
    "Volunteers arrive at 9:15 to set up\n"
    "Bring spare envelopes for small seed packets\n"
)

with open(f"{DEMO_DIR}/catalog.txt", "w") as f:
    f.write(catalog)

with open(f"{DEMO_DIR}/notes.txt", "w") as f:
    f.write(notes)


def sha256_of(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


ORIGINAL_HASHES = {
    "catalog.txt": sha256_of(f"{DEMO_DIR}/catalog.txt"),
    "notes.txt": sha256_of(f"{DEMO_DIR}/notes.txt"),
}

for name, digest in ORIGINAL_HASHES.items():
    print(f"{name}: {digest[:16]}...")

catalog.txt: a04c87fba0b55668...
notes.txt: 9c87faa1244b07bb...


## Cell 5: Verify the files

Before any agent runs, we confirm the ground truth: which files exist and what they say. This cell prints the directory tree, then the contents of each file.

It also checks whether any file mentions the word "table". The expected answer is `False`. When we later ask a session about the table number, we will know it could only have come from a prompt, never from a file.

In [ ]:
# Cell 5
import os

print(DEMO_DIR)
for name in sorted(os.listdir(DEMO_DIR)):
    print(f"  {name}")
print()

mentions_table = False

for name in sorted(os.listdir(DEMO_DIR)):
    with open(f"{DEMO_DIR}/{name}") as f:
        text = f.read()
    print(f"--- {name} ---")
    print(text)
    if "table" in text.lower():
        mentions_table = True

print("Any file mentions a table:", mentions_table)

/content/seedlibrary_demo
  catalog.txt
  notes.txt

--- catalog.txt ---
marigold: 40 packets
basil: 12 packets
okra: 75 packets
sunflower: 30 packets
cilantro: 22 packets
zinnia: 18 packets

--- notes.txt ---
Seed swap day notes
Doors open at 10:00
Volunteers arrive at 9:15 to set up
Bring spare envelopes for small seed packets

Any file mentions a table: False


## Cell 6: Define the prompts

We keep every prompt in a variable so the agent cells that follow stay short and readable.

| Variable | Used in | Purpose |
| :-- | :-- | :-- |
| `BASE_PROMPT_1` | Cell 7 | Base session, turn 1: read the notes and report the door time. |
| `BASE_PROMPT_2` | Cell 8 | Base session, turn 2: give the session a new fact, the table number. |
| `FORK_A_PROMPT` | Cell 9 | Fork A: write a short announcement to a file. |
| `FORK_B_PROMPT` | Cell 10 | Fork B: write a volunteer checklist to a file. |
| `ASK_ORIGINAL` | Cell 12 | Ask the original session whether it wrote anything. |
| `ASK_FORK_A` | Cell 12 | Ask Fork A what it wrote. |
| `READ_PROMPT` | Cell 13 | Ask the original session to read the announcement file. |
| `OFFLINE_PROMPT` | Cell 15 | Ask the offline fork what it knows. |

The file prompts use full paths, so the agent never has to guess where a file lives.

In [ ]:
# Cell 6
BASE_PROMPT_1 = (
    "Read /content/seedlibrary_demo/notes.txt "
    "and tell me what time doors open."
)

BASE_PROMPT_2 = (
    "Remember that the swap table number is 7. "
    "Just confirm."
)

FORK_A_PROMPT = (
    "Write a two-sentence swap-day announcement to "
    "/content/seedlibrary_demo/announcement.txt."
)

FORK_B_PROMPT = (
    "Write a short volunteer checklist of five to six items to "
    "/content/seedlibrary_demo/volunteer_checklist.txt."
)

ASK_ORIGINAL = (
    "Did you write any announcement or checklist in this conversation? "
    "Answer from this conversation only."
)

ASK_FORK_A = (
    "What did you write? "
    "Answer from this conversation only."
)

READ_PROMPT = (
    "Read /content/seedlibrary_demo/announcement.txt and tell me what it says. "
    "Then tell me whether you remember writing it."
)

OFFLINE_PROMPT = (
    "What time do doors open, "
    "and what table number did I mention?"
)

## Cell 7: Base session, turn 1

This starts the **base conversation**. Remember the mental model from earlier lectures: `query()` does not return a value. It returns an **async stream of messages** while the agent loop runs. The final answer, and the session ID, arrive on the `ResultMessage`.

| Piece | What it does |
| :-- | :-- |
| `allowed_tools=["Read"]` | The agent may read files, and nothing else. |
| `cwd="/content"` | Sets the working directory for the session. |
| `message.session_id` | The ID of this session. We save it as `BASE_ID`. |

After the run, the cell calls `get_session_messages(BASE_ID, directory="/content")`. This is a **synchronous** function that reads the saved transcript from disk. It returns a list of `SessionMessage` objects, each with a `type`, a `uuid`, a `session_id` and a `message`. We save the `uuid` of the last message as `FORK_POINT_ID`. That is a bookmark for the end of turn 1, and we will use it in Cell 14.

**What to look for:** the agent reads `notes.txt` and reports the door time. Then check the printed message type and UUID. The UUID identifies one specific message in the transcript.

In [ ]:
# Cell 7
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    ResultMessage,
    get_session_messages,
)

BASE_ID = None

async for message in query(
    prompt=BASE_PROMPT_1,
    options=ClaudeAgentOptions(
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    print(message)
    if isinstance(message, ResultMessage):
        BASE_ID = message.session_id
        if message.subtype == "success":
            print(message.result)

print()
print("BASE_ID:", BASE_ID)

messages = get_session_messages(BASE_ID, directory="/content")
FORK_POINT_ID = messages[-1].uuid

print("Messages in the base session:", len(messages))
print("Type of the last message:", messages[-1].type)
print("FORK_POINT_ID:", FORK_POINT_ID)

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/content', 'session_id': '0929ef06-fa8e-4199-9bad-f6387679a8e6', 'tools': ['Task', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'DesignSync', 'Edit', 'EnterWorktree', 'ExitWorktree', 'ListAgents', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'ReportFindings', 'ScheduleWakeup', 'SendMessage', 'Skill', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskStop', 'TaskUpdate', 'ToolSearch', 'WebFetch', 'WebSearch', 'Workflow', 'Write'], 'mcp_servers': [], 'model': 'claude-haiku-4-5', 'permissionMode': 'default', 'slash_commands': ['deep-research', 'design', 'design-sync', 'dataviz', 'update-config', 'verify', 'debug', 'code-review', 'simplify', 'batch', 'fewer-permission-prompts', 'doctor', 'loop', 'claude-api', 'workflow-authoring', 'run', 'run-skill-generator', 'plugin-authoring', 'advisor', 'agents', 'auto-mode-setup', 'autocompact', 'clear', 'color', 'compact', 'config', 'output-style', 'context', 'effor

## Cell 8: Base session, turn 2

Now we continue the base conversation with a plain **resume**. Passing `resume=BASE_ID` returns to the session with its full history, so the agent already knows about the door time.

The prompt gives the session one new fact: the swap table number is 7. This fact exists **only** in this prompt, not in any file.

The cell prints whether the session ID after a plain resume is still equal to `BASE_ID`, then updates `BASE_ID` from the result so we always hold the ID the SDK reported. Read that printed line rather than assuming the answer.

Finally, the cell calls `list_sessions(directory="/content")` to count the sessions saved so far. Keep that number in mind. It will change after the forks.

**What to look for:** a short confirmation of the table number, the ID comparison, and the session count.

In [ ]:
# Cell 8
from claude_agent_sdk import list_sessions

async for message in query(
    prompt=BASE_PROMPT_2,
    options=ClaudeAgentOptions(
        resume=BASE_ID,
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        print("Same session ID after a plain resume:", message.session_id == BASE_ID)
        BASE_ID = message.session_id
        if message.subtype == "success":
            print(message.result)

print()
print("Sessions so far:", len(list_sessions(directory="/content")))

Same session ID after a plain resume: True
Confirmed! I'll remember that the swap table number is 7.

Sessions so far: 1


## Cell 9: Fork A

This is the core of the lecture. We create a **fork**: a new session that starts with a copy of the base conversation's history and then diverges.

You fork by passing two options together in `ClaudeAgentOptions`:

| Option | Meaning |
| :-- | :-- |
| `resume=BASE_ID` | Which session to start from. |
| `fork_session=True` | Do not continue that session. Copy its history into a **new** session instead. |

These two options only make sense together. `fork_session=True` says "copy", and `resume` says "copy what". On its own, `fork_session=True` has no session to copy.

The fork gets its own session ID, which arrives on its `ResultMessage`. We save it as `FORK_A_ID`. The original session's ID and history stay unchanged.

Fork A is allowed to use the `Write` tool and nothing else. It will write a two-sentence announcement. It was never told the door time or the table number in this prompt, so any facts it uses come from the copied history.

> **Naming trap.** The name `fork_session` appears twice in the SDK, with two different meanings.
>
> | Where | Form | Meaning |
> | :-- | :-- | :-- |
> | `ClaudeAgentOptions` | `fork_session=True` | A **field**. Fork at resume time, while running the agent. |
> | `claude_agent_sdk` | `fork_session(...)` | A standalone **function**. Copy a transcript offline, without running the agent. |
>
> You will meet the function in Cell 14.

**What to look for:** the agent writes `announcement.txt` and reports what it wrote. Check whether the announcement uses facts from the base conversation.

In [ ]:
# Cell 9
FORK_A_ID = None

async for message in query(
    prompt=FORK_A_PROMPT,
    options=ClaudeAgentOptions(
        resume=BASE_ID,
        fork_session=True,
        allowed_tools=["Write"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        FORK_A_ID = message.session_id
        if message.subtype == "success":
            print(message.result)

print()
print("FORK_A_ID:", FORK_A_ID)

Done! I've written a two-sentence swap-day announcement to `/content/seedlibrary_demo/announcement.txt` that includes the door opening time (10:00) and the swap table number (7).

FORK_A_ID: 6c76818b-e20e-4975-9611-3d118c4f5500


## Cell 10: Fork B

We use the same pattern again from the **same starting point**: `resume=BASE_ID` together with `fork_session=True`. Only the prompt and the output file differ. This time the agent writes a volunteer checklist to `volunteer_checklist.txt`.

Because `BASE_ID` is unchanged by Fork A, Fork B starts from the identical history. Two forks, two different directions, one shared past. This is the "what if we did this differently?" workflow that resuming cannot give you.

**What to look for:** a checklist file, and a new session ID saved as `FORK_B_ID`.

In [ ]:
# Cell 10
FORK_B_ID = None

async for message in query(
    prompt=FORK_B_PROMPT,
    options=ClaudeAgentOptions(
        resume=BASE_ID,
        fork_session=True,
        allowed_tools=["Write"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        FORK_B_ID = message.session_id
        if message.subtype == "success":
            print(message.result)

print()
print("FORK_B_ID:", FORK_B_ID)

Done! I've created a volunteer checklist with 6 items that incorporates the details from the notes—the 9:15 arrival time, 10:00 door opening, spare envelopes for small packets, and Table 7. The checklist covers setup, supplies, and key tasks for a smooth seed swap event.

FORK_B_ID: 20aab869-c08f-4ce1-a175-0748408c33a2


## Cell 11: Ground truth

Now we check what actually happened, using facts rather than the agent's own words. This cell prints four things:

1. The three session IDs, and whether all three are different.
2. Whether `announcement.txt` and `volunteer_checklist.txt` exist, and what they contain.
3. How many sessions `list_sessions()` now reports for `/content`. Compare it with the count from Cell 8.
4. Whether the hashes of `catalog.txt` and `notes.txt` still match the ones we recorded in Cell 4.

**What to look for:** three different IDs, two new files that exist on disk, a higher session count, and both original files unchanged. Forking added new sessions and new files. It did not alter the original conversation or the original files.

In [ ]:
# Cell 11
import os

print("BASE_ID:  ", BASE_ID)
print("FORK_A_ID:", FORK_A_ID)
print("FORK_B_ID:", FORK_B_ID)
print("All three differ:", len({BASE_ID, FORK_A_ID, FORK_B_ID}) == 3)
print()

for name in ["announcement.txt", "volunteer_checklist.txt"]:
    path = f"{DEMO_DIR}/{name}"
    exists = os.path.exists(path)
    print(f"{name} exists:", exists)
    if exists:
        with open(path) as f:
            print(f.read())
    print()

sessions = list_sessions(directory="/content")
print("Sessions in /content:", len(sessions))
print()

for name in ["catalog.txt", "notes.txt"]:
    unchanged = sha256_of(f"{DEMO_DIR}/{name}") == ORIGINAL_HASHES[name]
    print(f"{name} unchanged:", unchanged)

BASE_ID:   0929ef06-fa8e-4199-9bad-f6387679a8e6
FORK_A_ID: 6c76818b-e20e-4975-9611-3d118c4f5500
FORK_B_ID: 20aab869-c08f-4ce1-a175-0748408c33a2
All three differ: True

announcement.txt exists: True
Join us for our seed swap day! Doors open at 10:00, and you'll find us at swap table number 7 where we're ready to exchange seeds and grow our community garden together.


volunteer_checklist.txt exists: True
Seed Swap Volunteer Checklist

□ Arrive by 9:15 AM for setup
□ Set up Table 7 with seed packets and labels
□ Stock spare envelopes for small seed packets
□ Test the check-in system and have sign-in sheets ready
□ Review seed swap guidelines and be ready to answer questions
□ Ensure doors are unlocked at 10:00 AM for opening


Sessions in /content: 3

catalog.txt unchanged: True
notes.txt unchanged: True


## Cell 12: Independence

Now the payoff for the branching idea. We resume two different sessions and ask each what it remembers.

| Session | Question | What it should reveal |
| :-- | :-- | :-- |
| The **original** (`BASE_ID`) | "Did you write any announcement or checklist?" | It has no memory of the detours. |
| **Fork A** (`FORK_A_ID`) | "What did you write?" | It knows the shared history plus its own branch. |

Two details keep this test fair:

- Both prompts say "Answer from this conversation only."
- `disallowed_tools=["Read", "Glob", "Grep"]` removes the read-only tools. Those tools are auto-approved by the CLI, so without removing them the agent could simply look at the files on disk and "discover" the answer. With them removed, the answer can only come from the session's own memory.

Here we resume with `resume=...` and **no** `fork_session`, so we are continuing each branch, not creating another one.

**What to look for:** the original should have nothing to report, while Fork A can describe its announcement. Same starting history, different memories. That is what independent branches look like.

In [ ]:
# Cell 12
print("--- Original session ---")
async for message in query(
    prompt=ASK_ORIGINAL,
    options=ClaudeAgentOptions(
        resume=BASE_ID,
        disallowed_tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

print()
print("--- Fork A ---")
async for message in query(
    prompt=ASK_FORK_A,
    options=ClaudeAgentOptions(
        resume=FORK_A_ID,
        disallowed_tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

--- Original session ---
No, I have not written any announcement or checklist in this conversation. I only read the notes.txt file and acknowledged that the swap table number is 7.

--- Fork A ---
I wrote:

"Join us for our seed swap day! Doors open at 10:00, and you'll find us at swap table number 7 where we're ready to exchange seeds and grow our community garden together."


## Cell 13: Files are shared

Forking branches the **conversation**, not your **files**. Fork A really wrote `announcement.txt` to disk, and that file is visible to every session working in the same directory, including the original.

To prove it, we resume the original session with the `Read` tool and ask it to read the announcement. We also ask whether it remembers writing it.

**What to look for:** the original can read and report the file's contents, but it should have no memory of writing it. The file is real. The memory belongs to Fork A.

That is the key limit of forking: **the agent's memory branches, your files do not.** If you need to branch and also undo file changes, you need a different tool, which is coming next.

In [ ]:
# Cell 13
async for message in query(
    prompt=READ_PROMPT,
    options=ClaudeAgentOptions(
        resume=BASE_ID,
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

**What the announcement says:**

"Join us for our seed swap day—doors open at 10:00 and we're ready to exchange seeds and gardening ideas! Visit us at swap table number 7 and bring your favorite seed packets to share."

**Do I remember writing it?**

No, I do not remember writing this announcement. I did not create or write this file during our conversation. It already existed when you asked me to read it.


## Cell 14: Offline fork with `fork_session()`

So far we forked **online**: the fork was created at the moment we resumed, while the agent ran. The SDK also offers an **offline** fork, the standalone `fork_session()` function. It copies a transcript into a new session **without running the agent** and returns a `ForkSessionResult` with a `.session_id`.

This is the second meaning of the name `fork_session`. It is a function you import, not the `ClaudeAgentOptions` field.

```
fork_session(session_id, directory=None, up_to_message_id=None, title=None)
```

| Parameter | Meaning |
| :-- | :-- |
| `session_id` | The session to copy. |
| `directory` | The project directory the session lives in. |
| `up_to_message_id` | Copy the transcript only up to and **including** this message. Leave it out to copy everything. |
| `title` | A readable title for the new session. |

We pass `up_to_message_id=FORK_POINT_ID`, the bookmark we saved at the end of turn 1 in Cell 7. That branches the conversation from **before** the table number was ever mentioned.

The function is synchronous, so there is no `await` here. It copies the transcript with fresh message IDs. It does not copy file history.

**The difference to remember:** online forking branches at the end of a conversation. Offline forking can branch at **any message**.

**What to look for:** a new session ID printed for the offline fork.

In [ ]:
# Cell 14
from claude_agent_sdk import fork_session

offline = fork_session(
    BASE_ID,
    directory="/content",
    up_to_message_id=FORK_POINT_ID,
    title="Before the table number",
)

OFFLINE_FORK_ID = offline.session_id
print("OFFLINE_FORK_ID:", OFFLINE_FORK_ID)

OFFLINE_FORK_ID: 561b2dc5-fe0a-4ce1-96b6-ad4401f7e05d


## Cell 15: Resume the offline fork

An offline fork is just a session on disk, so we use it like any other: pass its ID to `resume`. The question asks for two facts:

- the door time, which was settled in turn 1, **before** the fork point, and
- the table number, which was given in turn 2, **after** the fork point.

As in Cell 12, `disallowed_tools` removes the read-only tools so the answer can only come from the session's own history.

**What to look for:** the fork should be able to answer the door time. For the table number, it has no record of ever hearing one, because that message sits after the point where we branched. Same base conversation, but this branch was cut earlier.

In [ ]:
# Cell 15
async for message in query(
    prompt=OFFLINE_PROMPT,
    options=ClaudeAgentOptions(
        resume=OFFLINE_FORK_ID,
        disallowed_tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

Based on the notes.txt file I just read:

1. **Doors open at 10:00**
2. **No table number is mentioned** in the file

The file contains:
- Doors open at 10:00
- Volunteers arrive at 9:15 to set up
- Bring spare envelopes for small seed packets

There's no reference to a table number. Did you perhaps mention it in a different file, or would you like to add that information?


## Cell 16: Final verification

We finish with ground truth again. This cell lists every session in `/content`, labels the ones we created, and re-checks the hashes of `catalog.txt` and `notes.txt`.

**What to look for:** the base session, both online forks and the offline fork appear as **separate entries**, and the offline fork shows the title we gave it. Both original files still match their recorded hashes.

### What you learned

- **Resume** extends one conversation. **Fork** creates a new session with a copy of the history, and its own session ID.
- Fork online with `resume=<session_id>` plus `fork_session=True`. The fork's ID arrives on its `ResultMessage`.
- Branches are independent. The original knows nothing about a detour, and a fork knows the shared history plus its own branch.
- Forking branches the conversation, **not your files**.
- `fork_session=True` (an option) and `fork_session()` (a function) are two different things. The function copies a transcript offline and can branch from any earlier message with `up_to_message_id`.

In [ ]:
# Cell 16
KNOWN = {
    BASE_ID: "base",
    FORK_A_ID: "fork A",
    FORK_B_ID: "fork B",
    OFFLINE_FORK_ID: "offline fork",
}

sessions = list_sessions(directory="/content")
print("Sessions in /content:", len(sessions))
print()

for s in sessions:
    label = KNOWN.get(s.session_id, "other")
    print(f"{s.session_id[:8]}  {label:<12}  {s.summary}")

print()
for name in ["catalog.txt", "notes.txt"]:
    unchanged = sha256_of(f"{DEMO_DIR}/{name}") == ORIGINAL_HASHES[name]
    print(f"{name} unchanged:", unchanged)

Sessions in /content: 4

561b2dc5  offline fork  Before the table number
c79d12f4  base          Seedlibrary demo notes
6d492eff  fork A        Seedlibrary demo notes
5de8b0e8  fork B        Seedlibrary demo notes

catalog.txt unchanged: True
notes.txt unchanged: True
